In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# Grid & Domain Setup
Nx, Ny = 41, 41
Lx, Ly = 1.0, 1.0
dx = Lx / (Nx - 1)
dy = Ly / (Ny - 1)

Ra = 100000
Pr = 0.1

# Fluid & Physical Properties
nu = 1e-3          # Kinematic viscosity
alpha = nu / Pr    # Thermal diffusivity
beta = 1e-2        # Thermal expansion coefficient
g = 9.81           # Gravity
rho = 1.0          # Reference density
mu = nu * rho

Tc = 0.0
Th = (Ra * (mu**2)) / ((rho**2) * g * beta * (Ly**3)) + Tc
T0 = 0.5 * (Th + Tc)

# Time Integration & Stability
dt_visc = 0.25 / (nu * (1/dx**2 + 1/dy**2))
dt_therm = 0.25 / (alpha * (1/dx**2 + 1/dy**2))
dt = min(dt_visc, dt_therm)
nt = 1500
nit = 50  # PPE inner relaxation iterations

# Field Arrays
u = np.zeros((Ny, Nx))
v = np.zeros((Ny, Nx))
p = np.zeros((Ny, Nx))
T = np.full((Ny, Nx), T0)

# Temperature BC Initialization
T[:, 0] = Tc   # Left Cold Wall
T[:, -1] = Th  # Right Hot Wall

# Plotting & Animation Setup
X, Y = np.meshgrid(np.linspace(0, Lx, Nx), np.linspace(0, Ly, Ny))
fig, ax = plt.subplots(figsize=(6.5, 5))
writer = animation.PillowWriter(fps=15)

# Main Loop with Frame Capture
with writer.saving(fig, "buoyancy_flow_development.gif", dpi=100):
    for n in range(nt):
        un = u.copy()
        vn = v.copy()
        Tn = T.copy()

        # 1. Temperature Transport Update
        dTdx2 = (Tn[1:-1, 2:] - 2*Tn[1:-1, 1:-1] + Tn[1:-1, :-2]) / dx**2
        dTdy2 = (Tn[2:, 1:-1] - 2*Tn[1:-1, 1:-1] + Tn[:-2, 1:-1]) / dy**2
        dTdx = (Tn[1:-1, 2:] - Tn[1:-1, :-2]) / (2*dx)
        dTdy = (Tn[2:, 1:-1] - Tn[:-2, 1:-1]) / (2*dy)

        T[1:-1, 1:-1] = Tn[1:-1, 1:-1] + dt * (
            - un[1:-1, 1:-1] * dTdx - vn[1:-1, 1:-1] * dTdy + alpha * (dTdx2 + dTdy2)
        )

        # Temperature Boundary Conditions
        T[:, 0] = Tc          # Left (Cold)
        T[:, -1] = Th         # Right (Hot)
        T[0, :] = T[1, :]     # Bottom (Adiabatic)
        T[-1, :] = T[-2, :]   # Top (Adiabatic)

        # 2. Intermediate Velocity Prediction (u*, v*)
        u_star = un.copy()
        v_star = vn.copy()

        # u-momentum
        dudx2 = (un[1:-1, 2:] - 2*un[1:-1, 1:-1] + un[1:-1, :-2]) / dx**2
        dudy2 = (un[2:, 1:-1] - 2*un[1:-1, 1:-1] + un[:-2, 1:-1]) / dy**2
        dudx = (un[1:-1, 2:] - un[1:-1, :-2]) / (2*dx)
        dudy = (un[2:, 1:-1] - un[:-2, 1:-1]) / (2*dy)

        u_star[1:-1, 1:-1] = un[1:-1, 1:-1] + dt * (
            - un[1:-1, 1:-1] * dudx - vn[1:-1, 1:-1] * dudy + nu * (dudx2 + dudy2)
        )

        # v-momentum (with Buoyancy)
        dvdx2 = (vn[1:-1, 2:] - 2*vn[1:-1, 1:-1] + vn[1:-1, :-2]) / dx**2
        dvdy2 = (vn[2:, 1:-1] - 2*vn[1:-1, 1:-1] + vn[:-2, 1:-1]) / dy**2
        dvdx = (vn[1:-1, 2:] - vn[1:-1, :-2]) / (2*dx)
        dvdy = (vn[2:, 1:-1] - vn[:-2, 1:-1]) / (2*dy)

        buoyancy = g * beta * (T[1:-1, 1:-1] - T0)

        v_star[1:-1, 1:-1] = vn[1:-1, 1:-1] + dt * (
            - un[1:-1, 1:-1] * dvdx - vn[1:-1, 1:-1] * dvdy + nu * (dvdx2 + dvdy2) + buoyancy
        )

        # Velocity BCs for u*, v* (No-slip)
        u_star[0, :] = u_star[-1, :] = u_star[:, 0] = u_star[:, -1] = 0.0
        v_star[0, :] = v_star[-1, :] = v_star[:, 0] = v_star[:, -1] = 0.0

        # 3. Divergence Source Term for PPE
        b = np.zeros((Ny, Nx))
        b[1:-1, 1:-1] = (rho / dt) * (
            (u_star[1:-1, 2:] - u_star[1:-1, :-2]) / (2*dx) +
            (v_star[2:, 1:-1] - v_star[:-2, 1:-1]) / (2*dy)
        )

        # 4. Pressure Poisson Solver
        for it in range(nit):
            pn = p.copy()
            p[1:-1, 1:-1] = (
                (pn[1:-1, 2:] + pn[1:-1, :-2]) * dy**2 +
                (pn[2:, 1:-1] + pn[:-2, 1:-1]) * dx**2 -
                b[1:-1, 1:-1] * dx**2 * dy**2
            ) / (2 * (dx**2 + dy**2))

            # Neumann BCs for Pressure
            p[:, -1] = p[:, -2]  # Right
            p[:, 0] = p[:, 1]    # Left
            p[0, :] = p[1, :]    # Bottom
            p[-1, :] = p[-2, :]  # Top

        # 5. Velocity Field Correction
        u[1:-1, 1:-1] = u_star[1:-1, 1:-1] - (dt / rho) * (
            (p[1:-1, 2:] - p[1:-1, :-2]) / (2*dx)
        )
        v[1:-1, 1:-1] = v_star[1:-1, 1:-1] - (dt / rho) * (
            (p[2:, 1:-1] - p[:-2, 1:-1]) / (2*dy)
        )

        # No-slip BCs for u, v
        u[0, :] = u[-1, :] = u[:, 0] = u[:, -1] = 0.0
        v[0, :] = v[-1, :] = v[:, 0] = v[:, -1] = 0.0

        # Render and append frame every 15 iterations
        if n % 15 == 0:
            ax.clear()
            ax.contourf(X, Y, T, 25, cmap='coolwarm', vmin=Tc, vmax=Th)
            ax.streamplot(X, Y, u, v, color='k', linewidth=0.8, density=1.0)
            ax.set_title(f"Buoyancy Flow Evolution | Step: {n}/{nt}")
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            writer.grab_frame()

plt.close()